# Diabetes Prediction — LR, DT, SVM from Scratch + KNN, NB from sklearn
**Paper:** Rajendra & Latifi (2021), *Computer Methods and Programs in Biomedicine Update*

- LR, Decision Tree, SVM — implemented from scratch
- KNN and Naive Bayes — from sklearn (as per assignment)
- Two datasets: PIMA Indians (DS1) and Vanderbilt African-Americans Virginia (DS2)

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Dynamically find the project root containing 'src' to append to sys.path
current_dir = os.getcwd()
while current_dir and not os.path.exists(os.path.join(current_dir, 'src')):
    parent_dir = os.path.dirname(current_dir)
    if parent_dir == current_dir:
        break
    current_dir = parent_dir
if os.path.exists(os.path.join(current_dir, 'src')) and current_dir not in sys.path:
    sys.path.append(current_dir)
project_root = current_dir

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import copy

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from cvxopt import matrix, solvers

# Import custom modules from src
from src.utils import split_data
from src.evaluation import evaluate
from src.preprocessing import prepare_vand, impute_zero_values
from src.feature_selection import add_clinical_flags
from src.models import (
    LogReg,
    DecisionTree,
    SVM,
    majority_vote_cv,
    majority_vote_predict,
    stacking
)



---
## 1. Models from Scratch

### Logistic Regression

In [ ]:
# LogReg class is imported from src.models
print("LogReg imported successfully.")


### Decision Tree

In [ ]:
# DecisionTree class is imported from src.models
print("DecisionTree imported successfully.")


### SVM

In [ ]:
# SVM class is imported from src.models
print("SVM imported successfully.")


---
## 2. Utility Functions

In [ ]:
# split_data and evaluate are imported from src.utils and src.evaluation respectively
print("split_data and evaluate imported successfully.")


---
## 3. Dataset 1 — PIMA Indians Diabetes

In [ ]:
pima_path = os.path.join(project_root, 'data', 'pima_diabetes.csv')
pima = pd.read_csv(pima_path)
print(pima.shape)
pima.head()


DS1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(pima.corr(), annot=True, fmt='.2f', cmap='RdPu', ax=axes[0])
axes[0].set_title('Correlation — DS1')

counts = pima['Outcome'].value_counts().sort_index()
axes[1].bar(['No Diabetes', 'Diabetes'], counts.values, color=['#3d5a80', '#c1121f'])
axes[1].set_title('Class Distribution — DS1')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
print(f"diabetic: {(pima['Outcome']==1).sum()}  |  healthy: {(pima['Outcome']==0).sum()}")

### Preprocessing — DS1
Medical columns like Glucose and BMI can't physically be zero — those are missing values coded as 0. Replace them with the column mean.

In [ ]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
pima_clean = impute_zero_values(pima, zero_cols)
print("zeros remaining:", pima_clean[zero_cols].eq(0).sum().to_dict())


### Baseline — All 5 Models on DS1

In [ ]:
X1_tr, X1_te, y1_tr, y1_te = split_data(pima_clean, 'Outcome')

acc_d1_lr_base,  _ = evaluate("LR  (scratch)",  LogReg(),                          X1_tr, X1_te, y1_tr, y1_te)
acc_d1_dt_base,  _ = evaluate("DT  (scratch)",  DecisionTree(),                    X1_tr, X1_te, y1_tr, y1_te)
acc_d1_svm_base, _ = evaluate("SVM (scratch)",  SVM(),                             X1_tr, X1_te, y1_tr, y1_te)
acc_d1_knn_base, _ = evaluate("KNN (sklearn)",  KNeighborsClassifier(n_neighbors=5),X1_tr, X1_te, y1_tr, y1_te)
acc_d1_nb_base,  _ = evaluate("NB  (sklearn)",  GaussianNB(),                      X1_tr, X1_te, y1_tr, y1_te)

### Feature Engineering — DS1
Five binary clinical flags (NF1–NF5) based on known diabetes thresholds, then keep the 8 features most correlated with the outcome.

In [ ]:
# add_clinical_flags is imported from src.feature_selection

pima_nf = add_clinical_flags(pima_clean)

# pick top 8 features by absolute correlation with Outcome
corr   = pima_nf.corr()
top8   = corr['Outcome'].drop('Outcome').abs().sort_values(ascending=False).head(8).index.tolist()
print("selected features:", top8)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdPu', ax=axes[0])
axes[0].set_title('Correlation - All + NF1-NF5')
sns.heatmap(pima_nf[top8 + ['Outcome']].corr(), annot=True, fmt='.2f', cmap='RdPu', ax=axes[1])
axes[1].set_title('Correlation - Top 8 Features')
plt.tight_layout()
plt.show()


### After Feature Selection — All 5 Models on DS1

In [ ]:
pima_fs = pima_nf[top8 + ['Outcome']]
X1s_tr, X1s_te, y1s_tr, y1s_te = split_data(pima_fs, 'Outcome')

acc_d1_lr_fs,  lr1  = evaluate("LR  (scratch)",  LogReg(),                           X1s_tr, X1s_te, y1s_tr, y1s_te)
acc_d1_dt_fs,  dt1  = evaluate("DT  (scratch)",  DecisionTree(),                     X1s_tr, X1s_te, y1s_tr, y1s_te)
acc_d1_svm_fs, svm1 = evaluate("SVM (scratch)",  SVM(),                              X1s_tr, X1s_te, y1s_tr, y1s_te)
acc_d1_knn_fs, knn1 = evaluate("KNN (sklearn)",  KNeighborsClassifier(n_neighbors=5), X1s_tr, X1s_te, y1s_tr, y1s_te)
acc_d1_nb_fs,  nb1  = evaluate("NB  (sklearn)",  GaussianNB(),                       X1s_tr, X1s_te, y1s_tr, y1s_te)

### Ensemble — Majority Voting — DS1
All 5 models vote; majority wins. Evaluated with 10-fold cross-validation.

In [ ]:
# majority_vote_cv and majority_vote_predict are imported from src.models
models_d1 = [
    ('lr',  LogReg()),
    ('dt',  DecisionTree()),
    ('svm', SVM()),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('nb',  GaussianNB()),
]

X1s_arr    = np.array(X1s_tr, float)
y1s_arr    = np.array(y1s_tr)
X1s_te_arr = np.array(X1s_te, float)

cv_mv_d1 = majority_vote_cv(models_d1, X1s_arr, y1s_arr)
print(f"Majority Vote  CV-10: {cv_mv_d1:.4f}")

mv_preds_d1 = majority_vote_predict(models_d1, X1s_arr, y1s_arr, X1s_te_arr)
acc_d1_mv   = accuracy_score(np.array(y1s_te), mv_preds_d1)
print(f"Majority Vote  Test:  {acc_d1_mv:.4f}")


### Ensemble — Stacking — DS1
Base learners: DT, NB, KNN. Meta-learner: LR. OOF predictions from base models feed into LR.

In [ ]:
# stacking is imported from src.models
base_d1 = [('dt', DecisionTree()), ('nb', GaussianNB()), ('knn', KNeighborsClassifier(n_neighbors=5))]

stack_preds_d1 = stacking(base_d1, LogReg(), X1s_arr, y1s_arr.astype(int), X1s_te_arr)
acc_d1_stack   = accuracy_score(np.array(y1s_te), stack_preds_d1)
print(f"Stacking Test: {acc_d1_stack:.4f}")


---
## 4. Dataset 2 — Vanderbilt (African Americans, Virginia)

In [ ]:
vand_path = os.path.join(project_root, 'data', 'vanderbilt_diabetes.csv')
raw_vand = pd.read_csv(vand_path, sep=';', decimal=',')
print("raw shape:", raw_vand.shape)
raw_vand.head()


### Build DS2
`glyhb >= 7.0` is the standard clinical cutoff for diabetes. Also derive BMI and waist-hip ratio from the raw columns.

In [ ]:
# prepare_vand is imported from src.preprocessing
vand = prepare_vand(raw_vand)
print("shape:", vand.shape)
print(f"diabetic: {(vand['Diabetes']==1).sum()}  |  healthy: {(vand['Diabetes']==0).sum()}")
vand.head()


### EDA — DS2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(vand.corr(), annot=True, fmt='.2f', cmap='RdPu', ax=axes[0])
axes[0].set_title('Correlation — DS2')

counts2 = vand['Diabetes'].value_counts().sort_index()
axes[1].bar(['No Diabetes', 'Diabetes'], counts2.values, color=['#3d5a80', '#c1121f'])
axes[1].set_title('Class Distribution — DS2')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts2.values):
    axes[1].text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Preprocessing — DS2
Same zero-replacement strategy as DS1.

In [ ]:
num_cols  = vand.select_dtypes(include=[np.number]).columns.drop('Diabetes')
vand_clean = impute_zero_values(vand, num_cols)
print("done.")
vand_clean.head(2)


### Baseline — All 5 Models on DS2

In [ ]:
X2_tr, X2_te, y2_tr, y2_te = split_data(vand_clean, 'Diabetes')

acc_d2_lr_base,  _ = evaluate("LR  (scratch)",  LogReg(),                           X2_tr, X2_te, y2_tr, y2_te)
acc_d2_dt_base,  _ = evaluate("DT  (scratch)",  DecisionTree(),                     X2_tr, X2_te, y2_tr, y2_te)
acc_d2_svm_base, _ = evaluate("SVM (scratch)",  SVM(),                              X2_tr, X2_te, y2_tr, y2_te)
acc_d2_knn_base, _ = evaluate("KNN (sklearn)",  KNeighborsClassifier(n_neighbors=5), X2_tr, X2_te, y2_tr, y2_te)
acc_d2_nb_base,  _ = evaluate("NB  (sklearn)",  GaussianNB(),                       X2_tr, X2_te, y2_tr, y2_te)

### Univariate Feature Selection — DS2 (Correlation)
Chi-square test (SelectKBest). Top 8 features selected. MinMax scaling applied before chi2.

In [ ]:
X2_all = vand_clean.drop(columns=['Diabetes'])
y2_all = vand_clean['Diabetes']

scaler = MinMaxScaler()
X2_scaled = scaler.fit_transform(X2_all)

selector = SelectKBest(score_func=chi2, k=8)
selector.fit(X2_scaled, y2_all)

chi2_scores = pd.Series(selector.scores_, index=X2_all.columns).sort_values(ascending=False)
selected_features = chi2_scores.head(8).index.tolist()
print('Chi-square scores:\n', chi2_scores.round(2))
print('\nSelected features:', selected_features)

fig, ax = plt.subplots(figsize=(10, 4))
chi2_scores.sort_values().plot(kind='barh', ax=ax, color='#3d5a80')
ax.set_title('Chi-Square Scores per Feature — Dataset 2')
ax.axvline(chi2_scores.iloc[7], color='red', linestyle='--', label='k=8 cutoff')
ax.legend()
plt.tight_layout()
plt.show()

### After Feature Selection — All 5 Models on DS2

In [ ]:
selected_feats = selected_features
vand_fs = vand_clean[selected_feats + ['Diabetes']]
X2s_tr, X2s_te, y2s_tr, y2s_te = split_data(vand_fs, 'Diabetes')

acc_d2_lr_fs,  lr2  = evaluate("LR  (scratch)",  LogReg(),                           X2s_tr, X2s_te, y2s_tr, y2s_te)
acc_d2_dt_fs,  dt2  = evaluate("DT  (scratch)",  DecisionTree(),                     X2s_tr, X2s_te, y2s_tr, y2s_te)
acc_d2_svm_fs, svm2 = evaluate("SVM (scratch)",  SVM(),                              X2s_tr, X2s_te, y2s_tr, y2s_te)
acc_d2_knn_fs, knn2 = evaluate("KNN (sklearn)",  KNeighborsClassifier(n_neighbors=5), X2s_tr, X2s_te, y2s_tr, y2s_te)
acc_d2_nb_fs,  nb2  = evaluate("NB  (sklearn)",  GaussianNB(),                       X2s_tr, X2s_te, y2s_tr, y2s_te)


### Ensemble — Majority Voting — DS2

In [ ]:
X2s_arr    = np.array(X2s_tr, float)
y2s_arr    = np.array(y2s_tr)
X2s_te_arr = np.array(X2s_te, float)

models_d2 = [
    ('lr',  LogReg()),
    ('dt',  DecisionTree()),
    ('svm', SVM()),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('nb',  GaussianNB()),
]

cv_mv_d2  = majority_vote_cv(models_d2, X2s_arr, y2s_arr)
print(f"Majority Vote  CV-10: {cv_mv_d2:.4f}")

mv_preds_d2 = majority_vote_predict(models_d2, X2s_arr, y2s_arr, X2s_te_arr)
acc_d2_mv   = accuracy_score(np.array(y2s_te), mv_preds_d2)
print(f"Majority Vote  Test:  {acc_d2_mv:.4f}")

### Ensemble — Stacking — DS2

In [ ]:
base_d2 = [('dt', DecisionTree()), ('nb', GaussianNB()), ('knn', KNeighborsClassifier(n_neighbors=5))]

stack_preds_d2 = stacking(base_d2, LogReg(), X2s_arr, y2s_arr.astype(int), X2s_te_arr)
acc_d2_stack   = accuracy_score(np.array(y2s_te), stack_preds_d2)
print(f"Stacking Test: {acc_d2_stack:.4f}")

---
## 5. Results Summary

In [ ]:
techniques = [
    'LR — baseline',    'DT — baseline',    'SVM — baseline',   'KNN — baseline',   'NB — baseline',
    'LR — feat. sel.',  'DT — feat. sel.',  'SVM — feat. sel.', 'KNN — feat. sel.', 'NB — feat. sel.',
    'Majority Vote', 'Stacking',
]

d1_accs = [acc_d1_lr_base, acc_d1_dt_base, acc_d1_svm_base, acc_d1_knn_base, acc_d1_nb_base,
           acc_d1_lr_fs,   acc_d1_dt_fs,   acc_d1_svm_fs,   acc_d1_knn_fs,   acc_d1_nb_fs,
           acc_d1_mv,      acc_d1_stack]

d2_accs = [acc_d2_lr_base, acc_d2_dt_base, acc_d2_svm_base, acc_d2_knn_base, acc_d2_nb_base,
           acc_d2_lr_fs,   acc_d2_dt_fs,   acc_d2_svm_fs,   acc_d2_knn_fs,   acc_d2_nb_fs,
           acc_d2_mv,      acc_d2_stack]

summary = pd.DataFrame({
    'Technique':       techniques,
    'DS1 (PIMA)':      np.round(d1_accs, 4),
    'DS2 (Vanderbilt)': np.round(d2_accs, 4),
})
print(summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

palette     = ['#3d5a80']*5 + ['#457b9d']*5 + ['#c1121f', '#e07a5f']
short_labels = ['LR-base','DT-base','SVM-base','KNN-base','NB-base',
                'LR-fs',  'DT-fs',  'SVM-fs',  'KNN-fs',  'NB-fs',
                'MajVote','Stack']

for ax, accs, title in zip(axes, [d1_accs, d2_accs], ['DS1 — PIMA', 'DS2 — Vanderbilt']):
    bars = ax.bar(short_labels, accs, color=palette)
    ax.set_ylim(0, 1.12)
    ax.set_title(title, fontsize=13)
    ax.set_ylabel('Accuracy')
    ax.tick_params(axis='x', rotation=45)
    for bar, v in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()